In [4]:
from functions import extract_surface_lines_np
import numpy as np
import tensorflow as tf

ModuleNotFoundError: No module named 'functions'

In [6]:
# test function
with open('models/cone/a cone with a 10.00mm diameter and a 20.00mm height.brep', 'r', encoding='utf-8') as f:
    b = f.read()
    print(extract_surface_lines_np(b))

FileNotFoundError: [Errno 2] No such file or directory: 'models/cone/a cone with a 10.00mm diameter and a 20.00mm height.brep'

In [7]:
import os

shapes = ['cone', 'cube', 'cylinder', 'sphere', 'pipe', 'round_plate', 'rectangular_plate']

file_list = [os.listdir('models/' + shape) for shape in shapes]

X = []
y = []

for shape in shapes:
    y.append([shape]*len(os.listdir('models/' + shape)))

# flatten
file_list = [item for items in file_list for item in items]
y = [item for items in y for item in items]

for i in range(0, len(file_list)):
    with open('models/' + y[i] + '/' + file_list[i], 'r', encoding='utf-8') as f:
        brep_str = f.read()
        X.append(extract_surface_lines_np(brep_str))

FileNotFoundError: [Errno 2] No such file or directory: 'models/cone'

In [8]:
from numpy import float32
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder

maxlen = 1024
X_pad = pad_sequences(X, maxlen=maxlen, dtype='float32', padding='post', truncating='post', value=0.0)
X_pad = X_pad[..., None]

num_outputs = len(shapes)
le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_arr = np.asarray(y_encoded, dtype='int32')

X_train, X_test, y_train, y_test = train_test_split(X_pad, y_arr, test_size=0.2, random_state=3)

model = models.Sequential([
    layers.Input(shape=(maxlen, 1)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_outputs, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

NameError: name 'X' is not defined

In [ ]:
model_trained = model.fit(X_train, y_train, epochs=10, validation_split=0.1, batch_size=64)

Epoch 1/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.3770 - loss: 1.4974 - val_accuracy: 0.7679 - val_loss: 0.7565
Epoch 2/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9544 - loss: 0.5895 - val_accuracy: 1.0000 - val_loss: 0.4639
Epoch 3/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 0.3609 - val_accuracy: 1.0000 - val_loss: 0.3105
Epoch 4/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 0.2430 - val_accuracy: 1.0000 - val_loss: 0.2128
Epoch 5/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 0.1707 - val_accuracy: 1.0000 - val_loss: 0.1508
Epoch 6/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 0.1239 - val_accuracy: 1.0000 - val_loss: 0.1122
Epoch 7/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 0.0948 - val_accuracy: 1.0000 - val_loss: 0.0850
Epoch 8/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 0.0742 - val_accuracy: 1.0000 - val_loss: 0.0664
Epoch 9

In [5]:
# Predictions for neural network
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

NameError: name 'model' is not defined

In [ ]:
# Accuracy
accuracy = accuracy_score(y_test, y_pred_classes)
print(f"Accuracy: {accuracy:.4f}")

In [ ]:
# Classification Report
report = classification_report(y_test, y_pred_classes, target_names=shapes)
print("\nClassification Report:")
print(report)

In [ ]:
# Confusion Matrix
conf_mat = confusion_matrix(y_test, y_pred_classes)

plt.figure(figsize=(10, 6))
sns.heatmap(conf_mat, annot=False, cmap='Blues', fmt='d',
            xticklabels=shapes,
            yticklabels=shapes)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()